# Codebase Intelligence Agent — Stage 1 Prototype
## Baseline GitHub Repo Code RAG Chatbot with AST Chunking & LangGraph Memory

### Stage 1 Objectives:
1. **Repository Ingestion**: Clone any public GitHub repository URL or load a local repository path.
2. **Semantic Python AST Chunking**: Parse source code into function and class chunks preserving signatures, docstrings, and line numbers.
3. **Local Vector Search**: Embed code chunks locally via Hugging Face (`sentence-transformers`) and index in FAISS.
4. **Linear LangGraph Engine**: Orchestrate `retrieve -> generate` with state persistence (`MemorySaver` / `SqliteSaver`) for multi-turn chat.
5. **Inline Citations**: Synthesize grounded responses referencing exact `[file_path:start_line-end_line]` citations.

### Step 1: Environment & Dependency Imports

In [ ]:
import os
import ast
import shutil
import hashlib
import re
from pathlib import Path
from typing import List, Dict, Any, Optional, TypedDict
from concurrent.futures import ThreadPoolExecutor, as_completed
import dotenv

# LangChain, Multi-Language Text Splitters & LangGraph Imports
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from langchain_core.messages import HumanMessage, AIMessage, BaseMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from pydantic import BaseModel, Field

# Git Import
import git

# Load Environment Variables (.env)
dotenv.load_dotenv()

groq_api_key = os.getenv('GROQ_API_KEY')
if not groq_api_key:
    raise ValueError('GROQ_API_KEY not found in environment or .env file!')

print('Environment loaded successfully.')
print(f"LangSmith Tracing Active: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")


### Step 2: Initialize Groq LLM & Local Embedding Model
- **LLM**: Groq LPU (`qwen/qwen3.8-27b` or `openai/gpt-oss-120b`) for ultra-fast grounded generation.
- **Embeddings**: Local `sentence-transformers/all-MiniLM-L6-v2` (Runs locally on CPU/GPU, 0 cost, no API limits).

In [ ]:
# Initialize Groq Model
PRIMARY_MODEL = "qwen/qwen3.8-27b"
llm = ChatGroq(
    model=PRIMARY_MODEL,
    temperature=0.1,
    api_key=groq_api_key
)

# Initialize Local HuggingFace Embeddings
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    model_kwargs={"device": "cpu"}
)

print(f"LLM initialized: {PRIMARY_MODEL}")
print(f"Embeddings initialized: {EMBEDDING_MODEL_NAME}")

### Step 3: Repository Cloner & Discovery Engine
Supports cloning any remote GitHub URL into a local workspace directory, or pointing directly to an existing local repo.

In [ ]:
### Step 4: Universal Multi-Language Chunker (Python AST + Java / TS / Go / Rust / C++ Splitters)
- **Python**: Uses native `ast` to parse functions, async functions, classes, signatures, docstrings, and line spans.
- **Java, JS/TS, Go, Rust, C++, C#, Kotlin, Scala**: Uses syntax-aware `Language` grammar splitters.
- **Configs & Build Files**: `pom.xml`, `build.gradle`, `requirements.txt`, `Dockerfile`, `.yaml`, `.json`.

# Multi-Language Grammar Mapping
EXTENSION_TO_LANGUAGE = {
    # JVM Languages
    '.java': Language.JAVA,
    '.kt': Language.KOTLIN,
    '.kts': Language.KOTLIN,
    '.scala': Language.SCALA,
    # Python
    '.py': Language.PYTHON,
    # JS / TS Ecosystem
    '.js': Language.JS,
    '.jsx': Language.JS,
    '.ts': Language.TS,
    '.tsx': Language.TS,
    # Systems
    '.go': Language.GO,
    '.rs': Language.RUST,
    '.cpp': Language.CPP,
    '.c': Language.C,
    '.cs': Language.CSHARP,
    # Markup
    '.md': Language.MARKDOWN,
    '.html': Language.HTML,
}

LANGUAGE_SPLITTERS = {
    lang: RecursiveCharacterTextSplitter.from_language(
        language=lang, chunk_size=1200, chunk_overlap=200
    )
    for lang in set(EXTENSION_TO_LANGUAGE.values())
}

DEFAULT_TEXT_SPLITTER = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=150, separators=['\n\n', '\n', ' ', '']
)

class UniversalCodeChunker:
    """
    Multi-Language semantic parser for Python, Java, JS/TS, Go, Rust, C++, C#, and configs.
    """
    @staticmethod
    def chunk_python(content: str, relative_path: str) -> List[Document]:
        lines = content.splitlines()
        try:
            tree = ast.parse(content)
        except SyntaxError:
            return UniversalCodeChunker.chunk_generic(content, relative_path, Language.PYTHON)
            
        chunks: List[Document] = []
        module_doc = ast.get_docstring(tree)
        if module_doc:
            chunks.append(Document(
                page_content=f"File: {relative_path} (Module Doc)\n\n{module_doc}",
                metadata={'file_path': relative_path, 'chunk_type': 'module_doc', 'start_line': 1, 'end_line': len(module_doc.splitlines()), 'language': 'python', 'source_type': 'code'}
            ))
            
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                start_l = node.lineno
                end_l = getattr(node, 'end_lineno', start_l + len(ast.unparse(node).splitlines()))
                code_segment = '\n'.join(lines[start_l - 1:end_l])
                docstring = ast.get_docstring(node) or 'No docstring provided.'
                formatted = f"File: {relative_path} (Lines {start_l}-{end_l})\nFunction: {node.name}\nDocstring: {docstring}\n\nCode:\n{code_segment}"
                chunks.append(Document(
                    page_content=formatted,
                    metadata={'file_path': relative_path, 'chunk_type': 'function', 'name': node.name, 'start_line': start_l, 'end_line': end_l, 'language': 'python', 'source_type': 'code'}
                ))
            elif isinstance(node, ast.ClassDef):
                start_l = node.lineno
                end_l = getattr(node, 'end_lineno', start_l + len(ast.unparse(node).splitlines()))
                code_segment = '\n'.join(lines[start_l - 1:end_l])
                docstring = ast.get_docstring(node) or 'No docstring provided.'
                methods = [m.name for m in node.body if isinstance(m, (ast.FunctionDef, ast.AsyncFunctionDef))]
                formatted = f"File: {relative_path} (Lines {start_l}-{end_l})\nClass: {node.name}\nMethods: {', '.join(methods) if methods else 'None'}\nDocstring: {docstring}\n\nCode:\n{code_segment}"
                chunks.append(Document(
                    page_content=formatted,
                    metadata={'file_path': relative_path, 'chunk_type': 'class', 'name': node.name, 'start_line': start_l, 'end_line': end_l, 'language': 'python', 'source_type': 'code'}
                ))
        return chunks if chunks else UniversalCodeChunker.chunk_generic(content, relative_path, Language.PYTHON)

    @staticmethod
    def chunk_generic(content: str, relative_path: str, language: Language) -> List[Document]:
        splitter = LANGUAGE_SPLITTERS.get(language, DEFAULT_TEXT_SPLITTER)
        text_chunks = splitter.split_text(content)
        lines = content.splitlines()
        docs = []
        current_line = 1
        for idx, chunk in enumerate(text_chunks):
            chunk_lines = len(chunk.splitlines())
            start_l = current_line
            end_l = min(len(lines), start_l + chunk_lines - 1)
            current_line = max(1, end_l - 2)
            docs.append(Document(
                page_content=f"File: {relative_path} (Lines {start_l}-{end_l})\nLanguage: {language.value}\n\nCode:\n{chunk}",
                metadata={'file_path': relative_path, 'chunk_type': 'code_block', 'start_line': start_l, 'end_line': end_l, 'language': language.value, 'source_type': 'code'}
            ))
        return docs

    @classmethod
    def process_file(cls, file_path: Path, repo_root: Path) -> List[Document]:
        try:
            rel_path = file_path.relative_to(repo_root).as_posix()
            content = file_path.read_text(encoding='utf-8', errors='ignore')
            if not content.strip(): return []
            ext = file_path.suffix.lower()
            if ext == '.py': return cls.chunk_python(content, rel_path)
            elif ext in EXTENSION_TO_LANGUAGE: return cls.chunk_generic(content, rel_path, EXTENSION_TO_LANGUAGE[ext])
            return [Document(page_content=f"File: {rel_path}\n\n{c}", metadata={'file_path': rel_path, 'language': 'config'}) for c in DEFAULT_TEXT_SPLITTER.split_text(content)]
        except Exception:
            return []


In [ ]:
class ASTCodeChunker:
    """
    Parses Python source code using the AST library into semantic chunks
    (classes, standalone functions, module-level docstrings).
    """
    
    @staticmethod
    def chunk_python_file(file_path: Path, relative_path: str) -> List[Document]:
        try:
            content = file_path.read_text(encoding="utf-8", errors="ignore")
        except Exception as e:
            print(f"Warning: Could not read {file_path}: {e}")
            return []
        
        lines = content.splitlines()
        if not content.strip():
            return []
            
        try:
            tree = ast.parse(content, filename=str(file_path))
        except SyntaxError:
            # Fallback to naive block splitting if file has invalid Python syntax
            return ASTCodeChunker._fallback_chunk(content, relative_path, "code")
            
        chunks: List[Document] = []
        
        # 1. Module-level docstring or header
        module_doc = ast.get_docstring(tree)
        if module_doc:
            chunks.append(Document(
                page_content=f"# Module Docstring: {relative_path}\n\n{module_doc}",
                metadata={
                    "file_path": relative_path,
                    "chunk_type": "module_doc",
                    "name": relative_path,
                    "start_line": 1,
                    "end_line": len(module_doc.splitlines()),
                    "source_type": "code"
                }
            ))
            
        # 2. Extract Top-level Classes and Functions
        for node in tree.body:
            if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
                start_l = node.lineno
                end_l = getattr(node, "end_lineno", start_l + len(ast.unparse(node).splitlines()))
                code_segment = "\n".join(lines[start_l - 1:end_l])
                docstring = ast.get_docstring(node) or "No docstring provided."
                
                formatted_content = f"File: {relative_path} (Lines {start_l}-{end_l})\nFunction: {node.name}\nDocstring: {docstring}\n\nCode:\n{code_segment}"
                chunks.append(Document(
                    page_content=formatted_content,
                    metadata={
                        "file_path": relative_path,
                        "chunk_type": "function",
                        "name": node.name,
                        "start_line": start_l,
                        "end_line": end_l,
                        "source_type": "code"
                    }
                ))
            
            elif isinstance(node, ast.ClassDef):
                start_l = node.lineno
                end_l = getattr(node, "end_lineno", start_l + len(ast.unparse(node).splitlines()))
                code_segment = "\n".join(lines[start_l - 1:end_l])
                docstring = ast.get_docstring(node) or "No docstring provided."
                methods = [m.name for m in node.body if isinstance(m, (ast.FunctionDef, ast.AsyncFunctionDef))]
                
                formatted_content = f"File: {relative_path} (Lines {start_l}-{end_l})\nClass: {node.name}\nMethods: {', '.join(methods) if methods else 'None'}\nDocstring: {docstring}\n\nCode:\n{code_segment}"
                chunks.append(Document(
                    page_content=formatted_content,
                    metadata={
                        "file_path": relative_path,
                        "chunk_type": "class",
                        "name": node.name,
                        "start_line": start_l,
                        "end_line": end_l,
                        "source_type": "code"
                    }
                ))
        
        # If no functions or classes found (e.g. simple script), chunk the full file
        if not chunks:
            return ASTCodeChunker._fallback_chunk(content, relative_path, "code")
            
        return chunks

    @staticmethod
    def _fallback_chunk(content: str, relative_path: str, source_type: str = "config") -> List[Document]:
        """Line-based window chunker for non-Python or flat files."""
        lines = content.splitlines()
        if not lines:
            return []
            
        chunks: List[Document] = []
        step = 60
        overlap = 15
        
        for i in range(0, len(lines), step - overlap):
            chunk_lines = lines[i:i + step]
            start_l = i + 1
            end_l = min(i + step, len(lines))
            chunk_text = "\n".join(chunk_lines)
            
            chunks.append(Document(
                page_content=f"File: {relative_path} (Lines {start_l}-{end_l})\n\n{chunk_text}",
                metadata={
                    "file_path": relative_path,
                    "chunk_type": "text_block",
                    "name": relative_path,
                    "start_line": start_l,
                    "end_line": end_l,
                    "source_type": source_type
                }
            ))
            if end_l >= len(lines):
                break
        return chunks

IGNORE_DIRS = {'.git', 'venv', '.venv', '__pycache__', 'node_modules', '.idea', '.vscode', 'dist', 'build', 'target', 'bin', '.gradle', '.mvn'}
IGNORE_EXTENSIONS = {'.pyc', '.png', '.jpg', '.jpeg', '.gif', '.svg', '.ico', '.zip', '.tar', '.gz', '.exe', '.dll', '.so', '.class', '.jar'}
VALID_CONFIG_EXTENSIONS = {'.txt', '.toml', '.yaml', '.yml', '.json', '.sql', '.sh', '.env.example', '.xml', '.properties', '.gradle'}

def index_repository_parallel(repo_path: Path, max_workers: int = 16) -> FAISS:
    """
    Multi-threaded scanner that parses Java, Python, TS, Go, Rust, C++ in parallel and builds FAISS.
    """
    candidate_files = []
    for root, dirs, files in os.walk(repo_path):
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]
        for f in files:
            fp = Path(root) / f
            if fp.suffix.lower() not in IGNORE_EXTENSIONS and not f.startswith('.'):
                candidate_files.append(fp)
                
    print(f"Processing {len(candidate_files)} files using {max_workers} parallel threads...")
    all_documents = []
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = [executor.submit(UniversalCodeChunker.process_file, fp, repo_path) for fp in candidate_files]
        for future in as_completed(futures):
            res = future.result()
            if res:
                all_documents.extend(res)
                
    print(f"Total multi-language chunks generated: {len(all_documents)}")
    if not all_documents:
        raise ValueError("No valid source code or configuration files found to index!")
        
    print("Building FAISS vector index...")
    vectorstore = FAISS.from_documents(all_documents, embeddings)
    print("Vector index built successfully!")
    return vectorstore


In [ ]:
IGNORE_DIRS = {".git", "venv", ".venv", "__pycache__", "node_modules", ".idea", ".vscode", "dist", "build"}
IGNORE_EXTENSIONS = {".pyc", ".png", ".jpg", ".jpeg", ".gif", ".svg", ".ico", ".zip", ".tar", ".gz", ".exe", ".dll", ".so"}
VALID_TEXT_EXTENSIONS = {".py", ".md", ".txt", ".toml", ".yaml", ".yml", ".json", ".sql", ".sh", ".env.example"}

def index_repository(repo_path: Path) -> FAISS:
    """
    Walks the repository, applies AST and fallback chunkers, and builds a FAISS vector store.
    """
    all_documents: List[Document] = []
    print(f"Scanning repository at {repo_path}...")
    
    for root, dirs, files in os.walk(repo_path):
        # In-place filter out ignored directories
        dirs[:] = [d for d in dirs if d not in IGNORE_DIRS]
        
        for file in files:
            file_p = Path(root) / file
            ext = file_p.suffix.lower()
            
            if ext in IGNORE_EXTENSIONS or file.startswith("."):
                continue
                
            rel_path = file_p.relative_to(repo_path).as_posix()
            
            if ext == ".py":
                docs = ASTCodeChunker.chunk_python_file(file_p, rel_path)
                all_documents.extend(docs)
            elif ext in VALID_TEXT_EXTENSIONS or file_p.name in {"Dockerfile", "Makefile", "requirements.txt"}:
                try:
                    text = file_p.read_text(encoding="utf-8", errors="ignore")
                    docs = ASTCodeChunker._fallback_chunk(text, rel_path, source_type="config")
                    all_documents.extend(docs)
                except Exception as e:
                    print(f"Skipping {rel_path}: {e}")
                    
    print(f"Total chunks generated: {len(all_documents)}")
    if not all_documents:
        raise ValueError("No valid source code documents were found to index!")
        
    print("Building FAISS vector index (this may take a few seconds)...\c")
    vectorstore = FAISS.from_documents(all_documents, embeddings)
    print("\nVector index built successfully!")
    return vectorstore

### Step 6: Define LangGraph State & Prompts

In [ ]:
class Citation(BaseModel):
    file_path: str = Field(description="Path of the cited file.")
    lines: Optional[str] = Field(default=None, description="Line numbers cited e.g. L10-L45.")

class BaselineState(TypedDict):
    question: str
    retrieved_docs: List[Document]
    answer: str
    citations: List[str]

SYSTEM_PROMPT = """You are an expert codebase intelligence assistant explaining source code.
Answer the user's question using ONLY the provided code and configuration context.

Rules:
1. Every factual explanation must be grounded directly in the provided context.
2. Include inline citations in the format: `[file_path:Lstart-Lend]` whenever referring to functions, classes, or logic.
3. If the context does not contain sufficient details to answer, explicitly state what is missing instead of making assumptions.
4. Be concise, technically precise, and format code snippets in Markdown."""

### Step 7: Build Linear LangGraph Nodes (`retrieve -> generate`)

In [ ]:
# Global reference to vectorstore for retrieval node
ACTIVE_VECTORSTORE: Optional[FAISS] = None

def retrieve_node(state: BaselineState) -> Dict[str, Any]:
    """
    Retrieves the top-k most relevant code chunks from FAISS.
    """
    query = state["question"]
    if ACTIVE_VECTORSTORE is None:
        raise ValueError("Vectorstore has not been indexed yet!")
        
    docs = ACTIVE_VECTORSTORE.similarity_search(query, k=5)
    return {"retrieved_docs": docs}

def generate_node(state: BaselineState) -> Dict[str, Any]:
    """
    Synthesizes a grounded answer with inline citations.
    """
    question = state["question"]
    docs = state.get("retrieved_docs", [])
    
    context_parts = []
    citations = []
    for i, d in enumerate(docs, 1):
        fp = d.metadata.get("file_path", "unknown")
        s_line = d.metadata.get("start_line", "?")
        e_line = d.metadata.get("end_line", "?")
        citation_tag = f"{fp}:L{s_line}-L{e_line}"
        citations.append(citation_tag)
        context_parts.append(f"--- Context Document {i} [{citation_tag}] ---\n{d.page_content}")
        
    formatted_context = "\n\n".join(context_parts)
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Context:\n{formatted_context}\n\nQuestion:\n{question}"}
    ]
    
    response = llm.invoke(messages)
    return {
        "answer": response.content,
        "citations": list(set(citations))
    }

### Step 8: Assemble StateGraph & Add Checkpointer for Conversational Memory

In [ ]:
def build_code_rag_agent():
    """
    Compiles the baseline LangGraph agent with in-memory checkpointing.
    """
    builder = StateGraph(BaselineState)
    builder.add_node("retrieve", retrieve_node)
    builder.add_node("generate", generate_node)
    
    builder.add_edge(START, "retrieve")
    builder.add_edge("retrieve", "generate")
    builder.add_edge("generate", END)
    
    memory = MemorySaver()
    return builder.compile(checkpointer=memory)

agent_app = build_code_rag_agent()
print("LangGraph Baseline Agent compiled successfully.")

### Step 9: End-to-End Execution on Local or Target Repository
Let's index our current workspace repository and test querying it!

In [ ]:
# Index the current workspace directory
current_repo_path = Path(".").resolve()
ACTIVE_VECTORSTORE = index_repository(current_repo_path)

# Sample Query 1
config = {"configurable": {"thread_id": "session-1"}}
sample_query = "Explain the ASTCodeChunker class and how it extracts python functions and classes."

print(f"\n--- Running Query: '{sample_query}' ---\n")
result = agent_app.invoke({"question": sample_query, "retrieved_docs": [], "answer": "", "citations": []}, config=config)

print("### Answer:")
print(result["answer"])
print("\n### Citations:")
for c in result["citations"]:
    print(f" - [{c}]")

### Step 10: Interactive Chat Loop Function
Run this cell to start an interactive chat session in the notebook!

In [ ]:
def ask_repo(question: str, session_id: str = "default-session"):
    """
    Helper function to query the ingested codebase with persistent conversational state.
    """
    cfg = {"configurable": {"thread_id": session_id}}
    res = agent_app.invoke({"question": question, "retrieved_docs": [], "answer": "", "citations": []}, config=cfg)
    print("\n==================== RESPONSE ====================")
    print(res["answer"])
    print("\n--- Sources Cited ---")
    for c in res["citations"]:
        print(f"  * {c}")
    print("==================================================\n")

# Try another question:
ask_repo("What are the ignore directories and extensions in the indexer?")